In [0]:
# ==============================================================================
# PROJETO 2 - FASE 2: TRATAMENTO, TIPAGEM E CORREÇÃO DE FUSO HORÁRIO (SILVER)
# ==============================================================================
from pyspark.sql.functions import col, upper, trim, to_timestamp, from_utc_timestamp

print("Iniciando o processamento da Camada Silver com correção de fuso horário...")

# 1. Lendo os dados acumulados na Camada Bronze
df_bronze = spark.read.table("workspace.crypto_analytics.bronze_crypto")

# 2. Aplicando as transformações de engenharia de dados
df_silver = (df_bronze
    .withColumn("crypto_id", trim(col("id")))
    .withColumn("simbolo", upper(trim(col("symbol"))))
    .withColumn("nome", trim(col("name")))
    
    # PASSO CORRIGIDO: Convertemos para timestamp e ajustamos do UTC para o Horário de Brasília
    .withColumn("ultima_atualizacao_api", from_utc_timestamp(to_timestamp(col("last_updated")), "America/Sao_Paulo"))
    .withColumn("data_hora_extracao", from_utc_timestamp(col("coletado_em"), "America/Sao_Paulo"))
    
    .select(
        col("crypto_id"),
        col("simbolo"),
        col("nome"),
        col("current_price").alias("preco_atual_usd"),
        col("market_cap").alias("valor_de_mercado_usd"),
        col("total_volume").alias("volume_negociado_24h"),
        col("price_change_percentage_24h").alias("variacao_percentual_24h"),
        col("ultima_atualizacao_api"),
        col("data_hora_extracao")
    )
)

# 3. Salvando na Camada Silver (sobrescrevendo com os dados corrigidos)
df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.crypto_analytics.silver_crypto")

print("✓ Sucesso! Dados limpos e horários convertidos para o fuso de Brasília (America/Sao_Paulo).")